# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Accessing metadata safely
md = dataset.metadata
# Display key dataset metadata
print(f"{md.name}: {md.description}\n")
print(f"Croissant Schema Identifier (@id): {md.id}")
print(f"Version: {md.version}\nLicense: {md.license}\nPublished: {md.datePublished if hasattr(md, 'datePublished') else 'unknown'}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore record sets and fields using only @id references

print("Available record sets and their fields (by @id):\n")

record_sets = list(dataset.metadata.recordSets)
# Collect record set @id and their fields
for rs in record_sets:
    print(f"Record Set: {rs.id}")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"  Field: {field.id} (name: {field.name}, type: {field.dataType})")
    print()

if not record_sets:
    print("No record sets found in dataset metadata. Attempting to infer possible record sets from files...")
    # Sometimes the recordSet property may be empty but the schema includes files encoded under `distribution`/`dataFileObject`
    # List top-level distribution/files
    if hasattr(dataset.metadata, 'files'):
        for f in dataset.metadata.files:
            print(f"File: {f.id} - {f.name}")
    else:
        print("No files listed either. Please check the schema.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all record sets to DataFrames
from collections.abc import Iterable

dataframes = {}
# Find all recordSet @id values again
record_set_ids = [rs.id for rs in dataset.metadata.recordSets]

if not record_set_ids:
    # Manually extract available record set @id from files, if present
    print("Warning: No explicit record sets found. Check 'files' if available.")
else:
    for rs_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} rows from record set: {rs_id}")
            print(f"Columns: {df.columns.tolist()}")
            # Display first few records
            display(df.head())
        except Exception as e:
            print(f"Failed to load records for record set {rs_id}: {e}")

# For further analysis, select the first available record set (if available)
if record_set_ids:
    selected_record_set = record_set_ids[0]
    print(f"Using record set for subsequent analysis: {selected_record_set}")
else:
    selected_record_set = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Ensure we have at least one loaded table
import numpy as np

if selected_record_set is not None and selected_record_set in dataframes and not dataframes[selected_record_set].empty:
    df = dataframes[selected_record_set]
    # Automatically select a numeric field for demonstration
    numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_fields:
        # Attempt to infer numeric columns by converting
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                numeric_fields.append(col)
            except:
                continue
        numeric_fields = list(set(numeric_fields))
    if numeric_fields:
        numeric_field = numeric_fields[0]
        threshold = df[numeric_field].mean() if np.isfinite(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Find a categorical/grouping field
        group_field = next((c for c in df.columns if pd.api.types.is_string_dtype(df[c]) and c != numeric_field), None)
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df)
        else:
            print("No string/categorical field found for grouping.")
    else:
        print("No numeric field available for EDA analysis.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set is not None and selected_record_set in dataframes and not dataframes[selected_record_set].empty and numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # If a group field is available, show boxplot
    if group_field:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric fields or data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to use the `mlcroissant` library to load and explore the FAIR\^2 dataset: <br>
- Data sourced from: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json <br>

Key exploration steps included:
- Listing record sets and their field `@id`s.
- Loading tabular data directly indexed by record set `@id` into Pandas DataFrames.
- Performing simple EDA steps such as filtering, normalization, grouping, and visualization, always referencing fields by their `@id`s or structural names.

Next steps: For more detailed domain-specific analyses, you may wish to explore additional combinations of fields and groupings relevant to proteomics research.

Feel free to adapt this notebook as a general template for other Croissant-compliant datasets!